# Deconvolving Glutamate Transients in Stimulus Trains

## What the current iGluSnFR analysis pipeline actually does

---

**Context:** Rossi, Perrot, Huber, Poulain, Doussau, Valera & Isope — *Bouton-Specific Diversity of Glutamate Release from Single Axons*

---

Granule cells can fire rapidly enough that successive SF-iGluSnFR.S72A responses overlap. The fluorescence observed after pulse 5, for example, contains the response to pulse 5 **plus the remaining tails of pulses 1–4**. We therefore cannot read bouton-specific short-term plasticity directly from raw peak heights.

The current analysis is more than a single textbook NNLS solve. It combines a globally calibrated kinetic model, tri-exponential template variants, stimulus-locked temporal jitter, a sequential forward decomposition, a second pass that regularizes kinetic fractions along the train, and a separate noise-floor procedure for failure calls.

This notebook explains that complete workflow. The simulations are deliberately transparent, but the order of operations and the meaning of the reported amplitudes follow the current `iglusnfr_optimized` preset in `Feature_extraction/demo_single_file.py`.

### What you will learn

1. Why raw peaks overestimate later responses in a train
2. How one high-SNR kinetic template is calibrated from recut events
3. Why the current fit screens fast, slow, and superslow template fractions
4. How the sequential forward pass removes earlier tails before fitting the next pulse
5. What the second pass regularizes—and what it does **not** regularize
6. How reconstructed peaks become corrected Glu-T amplitudes and An/A1 profiles
7. How baseline fluctuations define failure thresholds and safe per-trial PPRs

> **Important distinction.** “NNLS” is retained as the name of the amplitude-extraction family, but the optimized preset uses a greedy sequence of non-negative one-component fits for the average trace. The simultaneous multi-column NNLS solver remains available as another fitting mode.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Set this to True when publication-ready PNG and SVG files are wanted.
SAVE_FIGURES = False
# Keep generated assets beside this publication notebook whether Jupyter was
# launched from the repository root or from this documentation directory.
LECTURE_DIR = Path("docs") / "nnls_lecture"
if not LECTURE_DIR.exists():
    LECTURE_DIR = Path(".")
FIGURE_DIR = LECTURE_DIR / "figures"

def finish_figure(fig, stem):
    """Apply final layout and optionally export a figure in raster and vector form."""
    fig.tight_layout()
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIGURE_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
        fig.savefig(FIGURE_DIR / f"{stem}.svg", bbox_inches="tight")
    plt.show()

print("Notebook helpers loaded.")

---
## Part 1: The complete path from fluorescence to STP

The analysis separates two jobs that are easy to confuse:

- **Kinetic calibration** asks what an individual SF-iGluSnFR response looks like.
- **Amplitude extraction** asks how much of that response occurred at each stimulus.

The average trace provides enough signal-to-noise to choose the kinetic fractions. Individual trials then reuse those average-derived shapes, allowing only amplitude and a small temporal shift to vary. This prevents noisy trials from inventing a different kinetic model for every pulse.

In [ ]:
# Figure 1 — current optimized workflow
fig, ax = plt.subplots(figsize=(15, 5.2))
ax.set_xlim(0, 15)
ax.set_ylim(0, 5.2)
ax.axis("off")

boxes = [
    (0.2, 3.2, 2.1, 1.1, "1  PREPROCESS", "interpolate NaNs\nbleach correction\nΔF/F₀"),
    (2.7, 3.2, 2.1, 1.1, "2  GLOBAL RECUT", "align events\nmean projection\n20× oversampling"),
    (5.2, 3.2, 2.1, 1.1, "3  CALIBRATE", "rise + fast/slow τ\nlast-tail superslow τ"),
    (7.7, 3.2, 2.1, 1.1, "4  AVERAGE FIT", "tri-exp variants\n±2 ms jitter\nsequential pass"),
    (10.2, 3.2, 2.1, 1.1, "5  SECOND PASS", "regularize slow and\nsuperslow fractions"),
    (12.7, 3.2, 2.1, 1.1, "6  REPORT", "reconstructed peaks\nsubtract prior tails\nAn/A1"),
]

for x, y, w, h, title, body in boxes:
    patch = plt.Rectangle((x, y), w, h, facecolor="#eef5fb", edgecolor="#245b78", lw=1.5)
    ax.add_patch(patch)
    ax.text(x + 0.12, y + h - 0.25, title, weight="bold", color="#17465f", va="top")
    ax.text(x + 0.12, y + h - 0.48, body, va="top", linespacing=1.25)

for i in range(len(boxes) - 1):
    x0 = boxes[i][0] + boxes[i][2]
    x1 = boxes[i + 1][0]
    ax.annotate("", xy=(x1 - 0.05, 3.75), xytext=(x0 + 0.05, 3.75),
                arrowprops=dict(arrowstyle="->", lw=1.5, color="#555555"))

trial_box = plt.Rectangle((7.7, 0.55), 4.6, 1.25, facecolor="#fff5df", edgecolor="#a96b16", lw=1.5)
ax.add_patch(trial_box)
ax.text(7.9, 1.52, "INDIVIDUAL TRIALS", weight="bold", color="#81500f", va="top")
ax.text(7.9, 1.25, "reuse average-derived fractions → robust forward fits with micro-shifts", va="top")
ax.annotate("", xy=(9.95, 1.82), xytext=(9.95, 3.18),
            arrowprops=dict(arrowstyle="<->", lw=1.4, color="#a96b16"))

null_box = plt.Rectangle((12.7, 0.55), 2.1, 1.25, facecolor="#f4ecf7", edgecolor="#70417d", lw=1.5)
ax.add_patch(null_box)
ax.text(12.88, 1.52, "NOISE FLOOR", weight="bold", color="#5c3267", va="top")
ax.text(12.88, 1.25, "Savgol baseline maxima\nthreshold + PPR floor", va="top")
ax.annotate("", xy=(13.75, 1.82), xytext=(13.75, 3.18),
            arrowprops=dict(arrowstyle="<->", lw=1.4, color="#70417d"))

ax.set_title("The current optimized analysis: kinetic selection on the average, constrained fitting on trials", pad=8)
finish_figure(fig, "01_current_workflow")

**Figure 1 | Overview of the optimized iGluSnFR train-analysis workflow.** Raw fluorescence traces are interpolated, corrected for bleaching, and expressed as ΔF/F₀ before stimulus-locked events are recut to calibrate the kinetic model. Tri-exponential variants and temporal jitter are selected on the average trace by sequential forward fitting, followed by a second pass that regularizes component fractions. Average-derived kinetics are then reused for individual trials, while corrected peak amplitudes and a separately estimated baseline noise floor provide the final An/A1 profiles.

---
## Part 2: Why raw peaks are not event amplitudes

For a simple exponential tail, the fraction remaining one inter-stimulus interval later is

$$\mathrm{residual}=e^{-\mathrm{ISI}/\tau}.$$

With $\tau=15$ ms, the remaining fraction is approximately **0.13% at 10 Hz**, **3.6% at 20 Hz**, and **26.4% at 50 Hz**. A tri-exponential sensor response can retain even more fluorescence because its slow and superslow components accumulate across the train.

The quantity to remove is therefore not a generic local baseline. It is the time-dependent sum of the modeled contributions from all earlier pulses.

In [ ]:
def tri_kernel(dt, tau_r=0.0014, tau_fast=0.006, tau_slow=0.022,
               tau_superslow=0.065, frac_slow=0.25, frac_superslow=0.08):
    """Peak-normalized rise × tri-exponential decay kernel for teaching figures."""
    dt = np.asarray(dt, float)
    tp = np.maximum(dt, 0.0)
    frac_slow = float(np.clip(frac_slow, 0.0, 1.0))
    frac_superslow = float(np.clip(frac_superslow, 0.0, 1.0))
    if frac_slow + frac_superslow > 1.0:
        frac_slow = 1.0 - frac_superslow
    frac_fast = 1.0 - frac_slow - frac_superslow
    rise = 1.0 - np.exp(-tp / tau_r)
    decay = (frac_fast * np.exp(-tp / tau_fast)
             + frac_slow * np.exp(-tp / tau_slow)
             + frac_superslow * np.exp(-tp / tau_superslow))
    k = rise * decay
    k[dt < 0] = 0.0
    peak = np.max(k)
    return k / peak if peak > 0 else k

dt = 0.0005
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, freq in zip(axes, [10, 20, 50]):
    isi_demo = 1.0 / freq
    stim_demo = 0.020 + np.arange(5) * isi_demo
    time_demo = np.arange(0, stim_demo[-1] + 0.120, dt)
    total = np.zeros_like(time_demo)
    for i, st in enumerate(stim_demo):
        event = tri_kernel(time_demo - st)
        total += event
        ax.plot(time_demo * 1000, event, color=plt.cm.viridis(0.15 + 0.16 * i), alpha=0.45, ls="--")
    ax.plot(time_demo * 1000, total, color="black", lw=2.2, label="recorded sum")
    for st in stim_demo:
        ax.axvline(st * 1000, color="#b23a48", lw=0.8, alpha=0.35)
    exp_residual = np.exp(-isi_demo / 0.015) * 100
    ax.set_title(f"{freq} Hz  |  exponential residual = {exp_residual:.1f}%")
    ax.set_xlabel("Time (ms)")
axes[0].set_ylabel("Normalized fluorescence")
axes[-1].legend(frameon=False)
fig.suptitle("Slow sensor components accumulate even when each event has the same amplitude", y=1.03)
finish_figure(fig, "02_overlap_by_frequency")

**Figure 2 | Frequency-dependent accumulation of overlapping sensor responses.** Dashed colored curves show five identical, isolated tri-exponential responses; black curves show their linear sum at 10, 20, and 50 Hz. Percentages report the residual expected from a 15-ms exponential component at the next stimulus. Slow and superslow components produce additional accumulation, making the measured peak increasingly different from the amplitude of the newly evoked event as stimulation frequency rises.

---
## Part 3: Calibrate kinetics before estimating amplitudes

The optimized preset first constructs a high-SNR **global recut event**:

1. Stimulus-locked event snippets are collected across trials.
2. They are projected with the mean and oversampled 20-fold.
3. A bi-exponential iGluSnFR model estimates the rise, fast-decay, and slow-decay kinetics.
4. In tri-exponential mode, the superslow time constant is estimated separately from the decay following the final pulse, where no subsequent event truncates the tail.

The recut fit determines the **time constants**. The train decomposition then screens the **fractions** assigned to fast, slow, and superslow components. Keeping those roles separate reduces the degeneracy between “a larger event” and “a slower event.”

In [ ]:
# Figure 3 — one calibrated set of time constants, many allowed kinetic mixtures
tk = np.arange(-0.005, 0.120, 0.0001)
rise = np.where(tk >= 0, 1 - np.exp(-np.maximum(tk, 0) / 0.0014), 0)
fast_component = rise * np.exp(-np.maximum(tk, 0) / 0.006)
slow_component = rise * np.exp(-np.maximum(tk, 0) / 0.022)
superslow_component = rise * np.exp(-np.maximum(tk, 0) / 0.065)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].plot(tk * 1000, fast_component, label="fast τ = 6 ms", lw=2)
axes[0].plot(tk * 1000, slow_component, label="slow τ = 22 ms", lw=2)
axes[0].plot(tk * 1000, superslow_component, label="superslow τ = 65 ms", lw=2)
axes[0].axvline(0, color="0.5", lw=0.8)
axes[0].set(xlabel="Time from event onset (ms)", ylabel="Component response",
            title="A) Calibrated kinetic components")
axes[0].legend(frameon=False)

mixtures = [(0.10, 0.00), (0.25, 0.05), (0.40, 0.15), (0.25, 0.35)]
for fs, fss in mixtures:
    axes[1].plot(tk * 1000, tri_kernel(tk, frac_slow=fs, frac_superslow=fss),
                 lw=2, label=f"slow={fs:.2f}, superslow={fss:.2f}")
axes[1].set(xlabel="Time from event onset (ms)", ylabel="Peak-normalized response",
            title="B) Variant library changes fractions, not calibrated τ values")
axes[1].legend(frameon=False, fontsize=9)
finish_figure(fig, "03_calibrated_components_and_variants")

**Figure 3 | Kinetic calibration is separated from event-wise mixture selection.** **A,** Fast, slow, and superslow response components generated from one calibrated set of time constants. **B,** Representative peak-normalized templates obtained by changing the relative component fractions while retaining those time constants. This separation limits the ambiguity between a larger event and an event with a longer decay.

---
## Part 4: Build the train one event at a time

The optimized preset uses `nnls_fit_mode="sequential"`. The same five operations are repeated for every stimulus:

1. **Construct the starting residual.** Before event $i$, subtract every component already accepted:

   $$r_i(t)=y(t)-\sum_{k<i}C_k(t).$$

2. **Open the event-specific fitting window.** For events 1–9, fitting stops 2 ms before the next stimulus. This prevents the next response from influencing the current event. Event 10 can use the longer post-train decay.
3. **Screen the variant library.** Candidate templates span slow fraction, superslow fraction, and stimulus-locked temporal jitter.
4. **Fit one non-negative scale for each candidate.** The candidate receives the amplitude that minimizes its weighted squared error. Savitzky–Golay-derived weights emphasize the response-bearing part of the window.
5. **Accept and carry forward.** The lowest-RMS candidate becomes $C_i(t)$. Its complete decay—not only the portion inside the fitting window—is added to the accumulated reconstruction and subtracted before event $i+1$.

### Event 1

No earlier component exists, so $r_1(t)=y(t)$. We screen the full variant library inside the first window, retain one template and amplitude, and call their product $C_1(t)$. The residual presented to event 2 is then

$$r_2(t)=y(t)-C_1(t).$$

### Event 2 and the rest of the train

Event 2 is therefore **not** fitted on the original fluorescence. It is fitted after the entire modeled tail of event 1 has been removed. Event 3 is fitted after removing events 1 and 2, and so on. This forward construction prevents accumulated slow fluorescence from being misidentified as new glutamate release.

In [ ]:
def sequential_variant_fit(y, time, stim_times, slow_grid, superslow_grid, jitter_ms,
                           tau_r=0.0014, tau_fast=0.006, tau_slow=0.022,
                           tau_superslow=0.065):
    """Transparent teaching implementation of the optimized average-trace forward pass."""
    dt_local = np.median(np.diff(time))
    window = max(5, int(round(0.015 / dt_local)))
    if window % 2 == 0:
        window += 1
    window = min(window, len(y) - (1 - len(y) % 2))
    y_sg = savgol_filter(y, window, 2)
    weights = np.abs(y_sg)
    weights -= np.min(weights)
    weights /= max(np.max(weights), 1e-12)

    accumulated = np.zeros_like(y)
    components = []
    amplitudes, selected_slow, selected_ss, selected_jitter = [], [], [], []
    residual_snapshots, accumulated_snapshots = [], []
    fit_masks, candidate_surfaces, selected_kernels = [], [], []

    for i, st in enumerate(stim_times):
        accumulated_snapshots.append(accumulated.copy())
        residual = y - accumulated
        residual_snapshots.append(residual.copy())
        end = stim_times[i + 1] - 0.002 if i < len(stim_times) - 1 else min(time[-1], st + 0.120)
        mask = (time >= st) & (time <= end)
        fit_masks.append(mask.copy())
        event_surface = np.full((len(superslow_grid), len(slow_grid)), np.nan)
        best = (np.inf, 0.0, None, 0.0, 0.0, 0.0)

        for jss, fss in enumerate(superslow_grid):
            for js, fs in enumerate(slow_grid):
                local_best_rms = np.inf
                for jitter in np.asarray(jitter_ms) / 1000.0:
                    k = tri_kernel(time - (st + jitter), tau_r, tau_fast, tau_slow,
                                   tau_superslow, fs, fss)
                    wk = weights[mask] * k[mask]
                    wr = weights[mask] * residual[mask]
                    denom = np.dot(wk, wk)
                    if denom <= 1e-12:
                        continue
                    amp = max(0.0, float(np.dot(wr, wk) / denom))
                    rms = float(np.sqrt(np.mean((wr - amp * wk) ** 2)))
                    local_best_rms = min(local_best_rms, rms)
                    if rms < best[0]:
                        best = (rms, amp, k, fs, fss, jitter)
                event_surface[jss, js] = local_best_rms

        _, amp, kernel, fs, fss, jitter = best
        component = amp * kernel
        accumulated += component
        components.append(component)
        amplitudes.append(amp)
        selected_slow.append(fs)
        selected_ss.append(fss)
        selected_jitter.append(jitter)
        candidate_surfaces.append(event_surface)
        selected_kernels.append(kernel)

    return {
        "amplitudes": np.asarray(amplitudes),
        "slow": np.asarray(selected_slow),
        "superslow": np.asarray(selected_ss),
        "jitter_s": np.asarray(selected_jitter),
        "components": components,
        "reconstruction": accumulated,
        "weights": weights,
        "residual_snapshots": residual_snapshots,
        "accumulated_snapshots": accumulated_snapshots,
        "fit_masks": fit_masks,
        "candidate_surfaces": candidate_surfaces,
        "selected_kernels": selected_kernels,
    }

# Synthetic 50 Hz train with known amplitudes and gradually changing mixtures
rng = np.random.default_rng(2026)
time = np.arange(0, 0.480, 0.0005)
stim_times = 0.080 + np.arange(10) * 0.020
true_amplitudes = np.array([1.00, 1.35, 1.52, 1.58, 1.50, 1.39, 1.28, 1.18, 1.10, 1.04])
true_slow = np.linspace(0.18, 0.38, 10)
true_superslow = np.linspace(0.02, 0.20, 10)

clean = np.zeros_like(time)
true_components = []
for st, amp, fs, fss in zip(stim_times, true_amplitudes, true_slow, true_superslow):
    comp = amp * tri_kernel(time - st, frac_slow=fs, frac_superslow=fss)
    clean += comp
    true_components.append(comp)
observed = clean + rng.normal(0, 0.045, time.size)

slow_grid = np.linspace(0.10, 0.50, 9)
superslow_grid = np.linspace(0.00, 0.30, 7)
jitter_grid_ms = np.arange(-2.0, 2.01, 0.5)
pass1 = sequential_variant_fit(observed, time, stim_times, slow_grid, superslow_grid, jitter_grid_ms)

In [ ]:
# Figure 4A — build Event 1 explicitly
event_idx = 0
st = stim_times[event_idx]
mask = pass1["fit_masks"][event_idx]
component = pass1["components"][event_idx]
residual_before = pass1["residual_snapshots"][event_idx]
residual_after = residual_before - component
selected_fs = pass1["slow"][event_idx]
selected_fss = pass1["superslow"][event_idx]
selected_jitter = pass1["jitter_s"][event_idx]
xlim_event1 = ((st - 0.010) * 1000, (stim_times[1] + 0.055) * 1000)

fig, axes = plt.subplots(2, 3, figsize=(16, 8.2))

axes[0, 0].plot(time * 1000, observed, color="0.55", lw=1.2, label="$y(t)$")
axes[0, 0].axvspan(time[mask][0] * 1000, time[mask][-1] * 1000,
                   color="#f4a261", alpha=0.22, label="Event 1 fit window")
axes[0, 0].axvline(st * 1000, color="#b23a48", lw=1)
axes[0, 0].set(xlim=xlim_event1, xlabel="Time (ms)", ylabel="ΔF/F₀ (a.u.)",
               title="A) Start: no earlier component exists")
axes[0, 0].legend(frameon=False, fontsize=9)

example_variants = [(0.10, 0.00), (0.30, 0.00), (0.30, 0.20), (0.50, 0.30)]
for fs, fss in example_variants:
    k = tri_kernel(time - st, frac_slow=fs, frac_superslow=fss)
    axes[0, 1].plot(time[mask] * 1000, k[mask], lw=1.1, alpha=0.7,
                    label=f"slow={fs:.2f}, super={fss:.2f}")
selected_kernel = pass1["selected_kernels"][event_idx]
axes[0, 1].plot(time[mask] * 1000, selected_kernel[mask], color="black", lw=2.5,
                label="selected shape")
axes[0, 1].set(xlabel="Time (ms)", ylabel="Unit-peak template",
               title="B) Screen fractions and ±2 ms jitter")
axes[0, 1].legend(frameon=False, fontsize=7)

im = axes[0, 2].imshow(pass1["candidate_surfaces"][event_idx], origin="lower",
                       aspect="auto", cmap="magma_r",
                       extent=[slow_grid[0], slow_grid[-1], superslow_grid[0], superslow_grid[-1]])
axes[0, 2].scatter(selected_fs, selected_fss, s=110, marker="x", color="white", lw=2.3)
axes[0, 2].set(xlabel="Slow fraction", ylabel="Superslow fraction",
               title="C) Keep the minimum weighted RMS")
fig.colorbar(im, ax=axes[0, 2], label="best RMS across jitters")

axes[1, 0].plot(time[mask] * 1000, residual_before[mask], color="0.45", lw=1.2,
                label="$r_1(t)=y(t)$")
axes[1, 0].plot(time[mask] * 1000, component[mask], color="#2a9d8f", lw=2.2,
                label="$C_1(t)=a_1K_1(t)$")
axes[1, 0].set(xlabel="Time (ms)", ylabel="ΔF/F₀ (a.u.)",
               title=f"D) Fit non-negative scale a₁={pass1['amplitudes'][0]:.2f}")
axes[1, 0].legend(frameon=False, fontsize=9)

axes[1, 1].plot(time * 1000, observed, color="0.72", lw=1, label="observed")
axes[1, 1].plot(time * 1000, component, color="#2a9d8f", lw=2.2, label="accepted $C_1(t)$")
axes[1, 1].axvline(stim_times[1] * 1000, color="#b23a48", lw=1, ls="--", label="Event 2 stimulus")
axes[1, 1].set(xlim=xlim_event1, xlabel="Time (ms)", ylabel="ΔF/F₀ (a.u.)",
               title="E) Carry the complete Event 1 tail forward")
axes[1, 1].legend(frameon=False, fontsize=9)

axes[1, 2].plot(time * 1000, observed, color="0.82", lw=1, label="original $y(t)$")
axes[1, 2].plot(time * 1000, residual_after, color="#3b528b", lw=1.7,
                label="$r_2(t)=y(t)-C_1(t)$")
axes[1, 2].axvline(stim_times[1] * 1000, color="#b23a48", lw=1)
axes[1, 2].set(xlim=xlim_event1, xlabel="Time (ms)", ylabel="Forward residual",
               title="F) This—not the raw trace—is fitted for Event 2")
axes[1, 2].legend(frameon=False, fontsize=9)

fig.suptitle("Building Event 1: choose one kinetic variant, scale it, and remove its full tail", y=1.01)
finish_figure(fig, "04a_build_event_1")

# Figure 4B — repeat the same construction for Event 2 and representative later events
shown_events = [1, 4, 9]  # Events 2, 5, and 10
fig = plt.figure(figsize=(17, 14))
gs = fig.add_gridspec(4, 4, height_ratios=[1, 1, 1, 1.25], hspace=0.48, wspace=0.34)

for row, idx in enumerate(shown_events):
    st = stim_times[idx]
    mask = pass1["fit_masks"][idx]
    accumulated_before = pass1["accumulated_snapshots"][idx]
    residual_before = pass1["residual_snapshots"][idx]
    component = pass1["components"][idx]
    residual_after = residual_before - component
    left = (stim_times[max(0, idx - 1)] - 0.008) * 1000
    right_anchor = stim_times[idx + 1] if idx < len(stim_times) - 1 else st + 0.100
    right = (right_anchor + 0.025) * 1000

    ax0 = fig.add_subplot(gs[row, 0])
    ax0.plot(time * 1000, observed, color="0.78", lw=1, label="observed")
    ax0.plot(time * 1000, accumulated_before, color="#f28e2b", lw=1.8,
             label=f"Σ accepted Events 1–{idx}" if idx else "none")
    ax0.set(xlim=(left, right), ylabel=f"Event {idx + 1}", title="1  Earlier components")
    if row == len(shown_events) - 1:
        ax0.set_xlabel("Time (ms)")
    ax0.legend(frameon=False, fontsize=7)

    ax1 = fig.add_subplot(gs[row, 1])
    ax1.plot(time * 1000, residual_before, color="#3b528b", lw=1.5)
    ax1.axvspan(time[mask][0] * 1000, time[mask][-1] * 1000, color="#f4a261", alpha=0.25)
    ax1.set(xlim=(left, right), title=f"2  Fit $r_{{{idx + 1}}}(t)$ in orange window")
    if row == len(shown_events) - 1:
        ax1.set_xlabel("Time (ms)")

    ax2 = fig.add_subplot(gs[row, 2])
    ax2.imshow(pass1["candidate_surfaces"][idx], origin="lower", aspect="auto", cmap="magma_r",
               extent=[slow_grid[0], slow_grid[-1], superslow_grid[0], superslow_grid[-1]])
    ax2.scatter(pass1["slow"][idx], pass1["superslow"][idx], s=75, marker="x", color="white", lw=2)
    ax2.set(title="3  Select variant", xlabel="Slow fraction", ylabel="Superslow fraction")

    ax3 = fig.add_subplot(gs[row, 3])
    ax3.plot(time * 1000, residual_before, color="0.72", lw=1, label="before")
    ax3.plot(time * 1000, component, color="#2a9d8f", lw=2, label=f"accepted $C_{{{idx + 1}}}$")
    ax3.plot(time * 1000, residual_after, color="#7a5195", lw=1.4, label="after subtraction")
    ax3.set(xlim=(left, right), title="4  Accept and update residual")
    if row == len(shown_events) - 1:
        ax3.set_xlabel("Time (ms)")
    ax3.legend(frameon=False, fontsize=7)

# Final row: the global result after all ten forward steps
ax_global = fig.add_subplot(gs[3, 0:2])
ax_global.plot(time * 1000, observed, color="0.72", lw=1, label="observed")
for i, comp in enumerate(pass1["components"]):
    ax_global.plot(time * 1000, comp, lw=1, color=plt.cm.viridis(0.07 + 0.09 * i), alpha=0.9)
ax_global.plot(time * 1000, pass1["reconstruction"], color="#b23a48", lw=2.3,
               label="Σ accepted components")
ax_global.set(xlim=(65, 380), xlabel="Time (ms)", ylabel="ΔF/F₀ (a.u.)",
              title="FINAL GLOBAL PROCESS  |  Ten accepted components reconstruct the train")
ax_global.legend(frameon=False, fontsize=8, ncol=2)

ax_amp = fig.add_subplot(gs[3, 2])
pulse = np.arange(1, len(stim_times) + 1)
ax_amp.plot(pulse, true_amplitudes, "--", color="0.4", label="simulation truth")
ax_amp.plot(pulse, pass1["amplitudes"], "o-", color="#2a9d8f", label="forward coefficients")
ax_amp.set(xlabel="Pulse", ylabel="Template scale", title="Recovered event scales")
ax_amp.legend(frameon=False, fontsize=8)

ax_final_resid = fig.add_subplot(gs[3, 3])
final_residual = observed - pass1["reconstruction"]
ax_final_resid.plot(time * 1000, final_residual, color="#7a5195", lw=1)
ax_final_resid.axhline(0, color="0.5", lw=0.8)
ax_final_resid.set(xlim=(65, 380), xlabel="Time (ms)", ylabel="Residual",
                   title="Final data − reconstruction")

fig.suptitle("The forward series: Event 2, Event 5, Event 10, then the complete decomposition", y=0.995)
finish_figure(fig, "04b_event_series_and_global_process")

**Figure 4 | Sequential construction of the stimulus-train model.** **A, Event 1 construction:** because no earlier component exists, the first residual equals the observed trace. Candidate slow/superslow fractions and temporal jitters are screened within the event-specific window; the minimum-RMS template is scaled non-negatively, accepted as C₁(t), and carried forward over its complete decay. Subtracting C₁(t) produces the residual presented to Event 2. **B, Forward series:** the same operation is shown for Events 2, 5, and 10. Each row displays the accumulated earlier components, the current fitting residual and window, the variant-selection surface, and the updated residual after accepting the event. The final row shows all ten components, their summed reconstruction, the recovered event scales, and the remaining global residual.

---
## Part 5: The second pass regularizes fractions along the train

The first pass can jump between neighboring variants because many slow/superslow mixtures produce similar traces. With template variants enabled, the current two-pass system regularizes the selected **fractions** across pulse number and refits amplitudes with those fractions fixed.

This is an important update from older descriptions of the method: in the optimized tri-exponential path, the second pass is not simply fitting an independently evolving decay constant for every pulse. The globally calibrated time constants remain the kinetic scaffold; slow and superslow fractions describe how the mixture evolves.

The figures below reuse the parameter-evolution visual language from `extract_metrics`. The stacked panel shows how fast, slow, and superslow fractions sum to one; the line panel shows each fraction independently. We first show the discrete variants selected by pass 1, then the regularized fractions supplied to pass 2.

The preset uses linear progression and `nnls_two_pass_guard=False`, so the smoothed second pass is accepted even if its RMS against noisy observations is slightly worse. The purpose is regularization and interpretability, not guaranteed training-error reduction.

For this teaching simulation, we also know the noise-free train used to generate the observations. The final comparison therefore reports **recovery RMS against that underlying train**. This makes the benefit of smoothing visible without suggesting that a constrained model must always fit noisy training data more closely.

In [ ]:
def robust_linear_progression(values, iterations=20, huber_delta=2.5):
    """Robust linear trend used here to illustrate the configured progression rule."""
    y = np.asarray(values, float)
    x = np.arange(y.size, dtype=float)
    X = np.column_stack([np.ones_like(x), x])
    weights = np.ones_like(y)
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    for _ in range(iterations):
        sw = np.sqrt(weights)
        beta = np.linalg.lstsq(X * sw[:, None], y * sw, rcond=None)[0]
        residual = y - X @ beta
        sigma = 1.4826 * np.median(np.abs(residual - np.median(residual))) + 1e-12
        scaled = np.abs(residual) / (huber_delta * sigma)
        weights = np.ones_like(scaled)
        outlier_mask = scaled > 1
        weights[outlier_mask] = 1.0 / scaled[outlier_mask]
    beta[1] = max(0.0, beta[1])
    return np.clip(X @ beta, 0.0, 1.0)

def refit_fixed_variants(y, time, stim_times, slow_values, superslow_values, jitter_ms):
    """Second forward pass: fractions fixed per event; amplitude and jitter remain free."""
    accumulated = np.zeros_like(y)
    components, amplitudes, shifts = [], [], []
    weights = pass1["weights"]
    for i, (st, fs, fss) in enumerate(zip(stim_times, slow_values, superslow_values)):
        residual = y - accumulated
        end = stim_times[i + 1] - 0.002 if i < len(stim_times) - 1 else min(time[-1], st + 0.120)
        mask = (time >= st) & (time <= end)
        best = (np.inf, 0.0, None, 0.0)
        for jitter in np.asarray(jitter_ms) / 1000.0:
            k = tri_kernel(time - (st + jitter), frac_slow=fs, frac_superslow=fss)
            wk, wr = weights[mask] * k[mask], weights[mask] * residual[mask]
            denom = np.dot(wk, wk)
            if denom <= 1e-12:
                continue
            amp = max(0.0, float(np.dot(wr, wk) / denom))
            rms = float(np.sqrt(np.mean((wr - amp * wk) ** 2)))
            if rms < best[0]:
                best = (rms, amp, k, jitter)
        _, amp, k, jitter = best
        component = amp * k
        accumulated += component
        components.append(component)
        amplitudes.append(amp)
        shifts.append(jitter)
    return {"amplitudes": np.asarray(amplitudes), "components": components,
            "reconstruction": accumulated, "jitter_s": np.asarray(shifts)}

slow_smoothed = robust_linear_progression(pass1["slow"])
ss_smoothed = robust_linear_progression(pass1["superslow"])
pass2 = refit_fixed_variants(observed, time, stim_times, slow_smoothed, ss_smoothed, jitter_grid_ms)

train_mask = (time >= stim_times[0]) & (time <= stim_times[-1] + 0.120)
# Training RMS measures agreement with the noisy observations. Because pass 1
# selects variants directly on those observations, regularization is not
# expected to improve this quantity systematically.
rms1_data = np.sqrt(np.mean((observed[train_mask] - pass1["reconstruction"][train_mask]) ** 2))
rms2_data = np.sqrt(np.mean((observed[train_mask] - pass2["reconstruction"][train_mask]) ** 2))
# In this simulation the underlying clean train is known. Recovery RMS therefore
# measures whether smoothing recovers the signal rather than fitting its noise.
rms1_recovery = np.sqrt(np.mean((clean[train_mask] - pass1["reconstruction"][train_mask]) ** 2))
rms2_recovery = np.sqrt(np.mean((clean[train_mask] - pass2["reconstruction"][train_mask]) ** 2))

def plot_component_fraction_evolution(frac_slow, frac_superslow, title, stem):
    """Reproduce the tri-exponential parameter-evolution view from extract_metrics."""
    frac_slow = np.asarray(frac_slow, float).copy()
    frac_superslow = np.asarray(frac_superslow, float).copy()
    total_slow = frac_slow + frac_superslow
    overfull = total_slow > 1.0
    frac_slow[overfull] /= total_slow[overfull]
    frac_superslow[overfull] /= total_slow[overfull]
    frac_fast = np.maximum(0.0, 1.0 - frac_slow - frac_superslow)
    event_indices = np.arange(1, frac_slow.size + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
    ax_stack, ax_lines = axes

    ax_stack.stackplot(
        event_indices, frac_fast, frac_slow, frac_superslow,
        labels=["frac_fast", "frac_slow", "frac_superslow"],
        colors=["tab:blue", "tab:orange", "tab:green"], alpha=0.70,
    )
    ax_stack.plot(event_indices, frac_fast, "o-", color="tab:blue", markersize=5, lw=1.5)
    ax_stack.plot(event_indices, frac_fast + frac_slow, "s-", color="tab:orange", markersize=5, lw=1.5)
    ax_stack.set(xlabel="Event #", ylabel="Cumulative fraction",
                 title="Component fractions (stacked)", ylim=(0, 1.05), xticks=event_indices)
    ax_stack.legend(loc="upper left", fontsize=8)
    ax_stack.grid(True, alpha=0.30)

    ax_lines.plot(event_indices, frac_fast, "o-", color="tab:blue",
                  label="frac_fast", markersize=6, lw=1.5)
    ax_lines.plot(event_indices, frac_slow, "s-", color="tab:orange",
                  label="frac_slow", markersize=6, lw=1.5)
    ax_lines.plot(event_indices, frac_superslow, "^-", color="tab:green",
                  label="frac_superslow", markersize=6, lw=1.5)
    ax_lines.set(xlabel="Event #", ylabel="Fraction",
                 title="Individual component fractions", ylim=(0, 1.05), xticks=event_indices)
    ax_lines.legend(loc="best", fontsize=8)
    ax_lines.grid(True, alpha=0.30)

    fig.suptitle(title, weight="bold", y=1.02)
    finish_figure(fig, stem)


# Figure 5A — raw discrete variants selected independently during pass 1
plot_component_fraction_evolution(
    pass1["slow"], pass1["superslow"],
    "Parameter evolution before smoothing (tri-exponential, pass 1)",
    "05a_parameter_evolution_unsmoothed",
)

# Figure 5B — fractions after the configured linear progression
plot_component_fraction_evolution(
    slow_smoothed, ss_smoothed,
    "Parameter evolution after smoothing (tri-exponential, pass 2)",
    "05b_parameter_evolution_smoothed",
)

# Figure 5C — smoothing regularizes fractions; amplitudes are then refitted
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
pulse = np.arange(1, len(stim_times) + 1)
axes[0].plot(pulse, pass1["slow"], "o", color="tab:orange", label="slow, pass 1")
axes[0].plot(pulse, slow_smoothed, "-s", color="#c65d00", label="slow, pass 2")
axes[0].plot(pulse, pass1["superslow"], "^", color="tab:green", label="superslow, pass 1")
axes[0].plot(pulse, ss_smoothed, "-^", color="#176b2c", label="superslow, pass 2")
axes[0].set(xlabel="Event #", ylabel="Fraction", title="A) Pass 1 selections → pass 2 progression",
            xticks=pulse, ylim=(0, 1.05))
axes[0].grid(True, alpha=0.3)
axes[0].legend(frameon=False, fontsize=8, ncol=2)

axes[1].plot(time * 1000, observed, color="0.78", lw=0.8, label="noisy observation")
axes[1].plot(time * 1000, clean, color="0.35", lw=1.2, ls="--", label="underlying clean train")
axes[1].plot(time * 1000, pass1["reconstruction"], lw=1.7,
             label=f"pass 1, recovery RMS={rms1_recovery:.4f}")
axes[1].plot(time * 1000, pass2["reconstruction"], lw=1.9,
             label=f"pass 2, recovery RMS={rms2_recovery:.4f}")
axes[1].set(xlim=(65, 330), xlabel="Time (ms)", ylabel="ΔF/F₀ (a.u.)",
            title="B) Regularization improves recovery of the underlying train")
axes[1].text(
    0.98, 0.04,
    f"noisy-data RMS: {rms1_data:.4f} → {rms2_data:.4f}",
    transform=axes[1].transAxes, ha="right", va="bottom", fontsize=8, color="0.35",
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=2),
)
axes[1].legend(frameon=False, fontsize=9)
finish_figure(fig, "05c_two_pass_refit_comparison")

**Figure 5 | Two-pass regularization of tri-exponential component fractions.** **A,** Parameter evolution before smoothing: discrete pass-1 selections are shown as cumulative stacked fractions and as individual fast, slow, and superslow trajectories. **B,** The same representation after applying the configured linear progression, which removes implausible event-to-event jumps while preserving the unit-sum constraint. **C,** Direct comparison of pass-1 selections and pass-2 fractions, followed by amplitude refitting with the regularized templates. Recovery RMS is calculated against the known noise-free train in this simulation; noisy-data RMS is reported separately because regularization is not expected to reduce training error systematically.

---
## Part 6: What is finally called an “amplitude”

The internal non-negative coefficient scales a template, but the reported amplitude series is defined from peaks:

1. Find the local maximum on the NNLS reconstruction in the stimulus-locked peak window.
2. At that peak time, calculate the summed contribution of all **earlier** fitted components.
3. Subtract that modeled carry-over from the reconstructed peak.

$$A_i^{\mathrm{corr}}=\hat y(t_i^{\mathrm{peak}})-\sum_{k<i}C_k(t_i^{\mathrm{peak}}).$$

The first event is unchanged because there is no earlier component. Later events can differ strongly from raw or uncorrected peaks at 50 Hz. The STP profile is then $A_i^{\mathrm{corr}}/A_1^{\mathrm{corr}}$.

In [ ]:
def reconstructed_peak_amplitudes(time, reconstruction, stim_times, components, peak_window_s=0.019):
    uncorrected, corrected, peak_times, prior_at_peak = [], [], [], []
    cumulative_previous = np.zeros_like(reconstruction)
    for i, st in enumerate(stim_times):
        mask = (time >= st - 0.001) & (time <= st + peak_window_s)
        local_idx = np.flatnonzero(mask)[np.argmax(reconstruction[mask])]
        peak = reconstruction[local_idx]
        baseline = cumulative_previous[local_idx]
        uncorrected.append(peak)
        corrected.append(peak - baseline)
        prior_at_peak.append(baseline)
        peak_times.append(time[local_idx])
        cumulative_previous += components[i]
    return map(np.asarray, (uncorrected, corrected, peak_times, prior_at_peak))

amp_peak, amp_corr, peak_times, prior_peaks = reconstructed_peak_amplitudes(
    time, pass2["reconstruction"], stim_times, pass2["components"]
)
raw_peaks = []
for st in stim_times:
    mask = (time >= st - 0.001) & (time <= st + 0.019)
    raw_peaks.append(np.max(observed[mask]))
raw_peaks = np.asarray(raw_peaks)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(time * 1000, observed, color="0.75", lw=1, label="observed")
axes[0].plot(time * 1000, pass2["reconstruction"], color="#245b78", lw=2, label="reconstruction")
axes[0].scatter(peak_times * 1000, amp_peak, color="#b23a48", zorder=4, label="reconstructed peak")
axes[0].scatter(peak_times * 1000, prior_peaks, facecolors="white", edgecolors="#b23a48",
                zorder=4, label="earlier components at peak")
for x, y0, y1 in zip(peak_times * 1000, prior_peaks, amp_peak):
    axes[0].plot([x, x], [y0, y1], color="#b23a48", lw=1.2)
axes[0].set(xlim=(65, 290), xlabel="Time (ms)", ylabel="ΔF/F₀ (a.u.)",
            title="A) Corrected amplitude = red dot minus white dot")
axes[0].legend(frameon=False, fontsize=8)

pulse = np.arange(1, len(stim_times) + 1)
# Plot supporting curves first, then the final result on top.
axes[1].plot(pulse, raw_peaks / raw_peaks[0], "o-", color="0.60", lw=1.4,
             label="UNCORRECTED — raw fluorescence peaks")
axes[1].plot(pulse, amp_peak / amp_peak[0], "s-", color="#f28e2b", lw=1.5,
             label="INTERMEDIATE — model peaks before tail subtraction")
axes[1].plot(pulse, true_amplitudes / true_amplitudes[0], "--", color="#1f2933", lw=1.8,
             label="GROUND TRUTH — simulated event amplitudes")
axes[1].plot(pulse, amp_corr / amp_corr[0], "o-", color="#168f83", lw=3.0, markersize=6,
             zorder=5, label="FINAL RESULT — prior-tail-corrected An/A1")
axes[1].axhline(1, color="0.75", lw=0.8)
axes[1].set(xlabel="Pulse", ylabel="Normalized amplitude",
            title="B) Prior-tail correction recovers the underlying STP profile")
handles, labels = axes[1].get_legend_handles_labels()
legend_order = [3, 2, 0, 1]  # final result, truth, uncorrected, intermediate
axes[1].legend(
    [handles[i] for i in legend_order], [labels[i] for i in legend_order],
    title="Curve meaning", loc="upper right", fontsize=8.5, title_fontsize=9,
    frameon=True, facecolor="white", edgecolor="0.85", framealpha=0.95,
)
finish_figure(fig, "06_peak_based_corrected_amplitudes")

**Figure 6 | Extraction of prior-tail-corrected event amplitudes.** **A,** For each stimulus, the local maximum of the reconstructed trace is identified (filled red circles) and the summed contribution of all earlier fitted components is evaluated at the same time (open circles). Their vertical difference defines the corrected amplitude assigned to the current event. **B,** The thick teal curve is the **final reported result**; the dark dashed curve is the known **simulation ground truth**. Gray raw-fluorescence peaks are the **uncorrected measurement**, whereas orange reconstruction peaks are an **intermediate model quantity before prior-tail subtraction**. The final correction closely recovers the underlying short-term-plasticity trajectory.

---
## Part 7: Individual trials and the noise floor

After kinetic fractions are selected on the average trace, each trial is fitted forward with those fractions fixed. The per-trial stage retains robust Huber reweighting and small stimulus-locked shifts. This shares kinetic information across trials without forcing identical release amplitudes.

Failure detection is a separate measurement problem. In the current optimized preset:

- `measurement="NNLS"`: NNLS-derived amplitudes are summarized and normalized.
- `fail_method="SAVGOL"`: the failure threshold comes from maxima measured on Savitzky–Golay-smoothed pre-train baseline segments.
- `threshold_mode="auto"`: SAVGOL selects a mean + SD threshold.
- `null_N=1.0`: the threshold is mean + 1 SD for this preset.
- `amplitude_floor_to_noise=True`: when an uncorrected per-trial amplitude is below threshold, its **corrected** counterpart is floored before corrected-PPR calculation. The uncorrected series remains untouched so floored events stay identifiable. Average amplitudes are not floored because `average_amplitude_floor_to_noise=False`.

The code also retains an NNLS-based baseline null distribution for diagnostics, but it is not the distribution driving failure calls under this preset.

In [ ]:
# Figure 7 — SAVGOL baseline null, threshold, and per-trial PPR protection
baseline_time = np.arange(-1.0, 0.0, 0.0005)
baseline_raw = rng.normal(0, 0.040, baseline_time.size)
baseline_sg = savgol_filter(baseline_raw, 9, 2)
candidate_starts = np.linspace(-0.95, -0.05, 180)
null_maxima = []
for start in candidate_starts:
    mask = (baseline_time >= start) & (baseline_time <= start + 0.019)
    null_maxima.append(np.max(baseline_sg[mask]))
null_maxima = np.asarray(null_maxima)
threshold = np.mean(null_maxima) + np.std(null_maxima)

example_trial_amps = np.array([0.035, 0.080, 0.115, 0.092, 0.071, 0.060, 0.055, 0.048, 0.043, 0.040])
# In the real pipeline this floor is applied to the corrected series only when
# the corresponding uncorrected amplitude is below threshold.
floored_trial_amps = np.maximum(example_trial_amps, threshold)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].plot(baseline_time, baseline_raw, color="0.75", lw=0.8, label="raw baseline")
axes[0].plot(baseline_time, baseline_sg, color="#5c3267", lw=1.5, label="Savgol baseline")
axes[0].set(xlabel="Time before train (s)", ylabel="ΔF/F₀", title="A) Pre-train baseline")
axes[0].legend(frameon=False)

axes[1].hist(null_maxima, bins=24, color="#9b72aa", alpha=0.8, edgecolor="white")
axes[1].axvline(threshold, color="#b23a48", lw=2, label=f"mean + 1 SD = {threshold:.3f}")
axes[1].set(xlabel="Windowed baseline maximum", ylabel="Count", title="B) SAVGOL null distribution")
axes[1].legend(frameon=False)

pulse = np.arange(1, 11)
axes[2].plot(pulse, example_trial_amps / example_trial_amps[0], "o--", color="0.6", label="unfloored PPR")
axes[2].plot(pulse, floored_trial_amps / floored_trial_amps[0], "o-", color="#2a9d8f", label="noise-floored PPR")
axes[2].axhline(1, color="0.75", lw=0.8)
axes[2].set(xlabel="Pulse", ylabel="An/A1", title="C) Flooring prevents division by a near-zero A1")
axes[2].legend(frameon=False)
finish_figure(fig, "07_null_threshold_and_ppr_floor")

**Figure 7 | Baseline-derived response threshold and protection of per-trial PPR estimates.** **A,** Raw and Savitzky–Golay-smoothed fluorescence during the pre-train baseline. **B,** Distribution of stimulus-sized maxima sampled from the smoothed baseline; under the optimized preset, the response threshold is the null mean plus one standard deviation. **C,** Illustration of corrected-amplitude flooring when the corresponding uncorrected event falls below threshold. This procedure prevents division by a near-zero A₁ while retaining the unfloored series for identification of noise-limited events.

---
## Part 8: Current preset versus configurable alternatives

| Analysis choice | Current `iglusnfr_optimized` preset | Available alternative |
|---|---|---|
| Event model | `iglusnfr_tri` | double exponential, bi-exponential iGluSnFR, other library models |
| Kinetic source | global recut | average or individual |
| Average fitting mode | sequential | simultaneous multi-column NNLS |
| Average weights | absolute Savitzky–Golay trace | uniform, linear, exponential, peak |
| Variant grid | slow × superslow fractions | disabled or model-specific grids |
| Temporal variants | −2 to +2 ms in 0.5 ms steps | custom grid or disabled |
| Progression | linear, two-pass fractions | fixed, free monotonic, none |
| Variant selection | winner per event in sequential mode | soft/hard selection in simultaneous mode |
| Failure threshold | SAVGOL null, mean + 1 SD | NNLS or raw null; MAD or SD |
| Per-trial floor | enabled | disabled or configured separately |

These distinctions matter when describing the method in the paper. A phrase such as “we used NNLS deconvolution” is directionally correct but incomplete. The result depends on how kinetic shapes are calibrated, how variants are selected, how overlap is removed, and how the noise floor is applied.

In [ ]:
print("=" * 72)
print("End of the current NNLS / sequential-decomposition lecture")
print("=" * 72)
print("Implementation: Feature_extraction/extract_metrics.py")
print("Current single-file preset: Feature_extraction/demo_single_file.py")
print("Residual diagnostics: Feature_extraction/demo_residual_diagnostics.py")
print("Set SAVE_FIGURES=True near the top to export every figure as PNG and SVG.")